# Companion notebook — `src/data/acquisition.py`

**Graduated:** `download_nass_yields` (Week 1, 2026-07-07) and `download_weather_data`
(Week 1, 2026-07-08).

Runs fully offline on the bundled samples under `data/samples/` — no API key, no
network. Executed in CI by the LINV-010 nbmake gate.

In [ ]:
# Bootstrap: make `src.*` importable and locate the repo root regardless of the
# directory this notebook is executed from (nbmake runs it from notebooks/data/).
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().resolve().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

## 1. Motivation

This repo's *volume track* needs thousands of county-year yield observations plus the
weather that drove them, to make hierarchical partial pooling and convergence
diagnostics visible (Phase 2, milestone M2a). Two sources:

- **USDA NASS Quick Stats** — annual county-level survey yields (corn, soybeans).
- **NASA POWER** — daily temperature/humidity/solar/wind for any point on Earth,
  keyless. *Curriculum decision:* POWER over NOAA/PRISM, because it is the same source
  the keragita-farm-intelligence platform consumes — this exact function is reused for
  the Kilifi capstone (Gongoni cell) in Phase 4.

The production platform tags every record with its `data_source` (its INV-006 — *no
untagged data path exists*). This module mirrors that habit from the first byte:
`api_nass` and `api_nasa_power` rows are distinguishable forever after. Governance
cards: `docs/data_cards/DC-usda-nass-quickstats.md`, `DC-nasapower-weather.md`.

## 2. The data semantics (this module's "mathematics")

No formulas — acquisition is about *provenance semantics*, and getting these wrong
poisons every model downstream:

**NASS:** query anatomy (`commodity_desc`, `year__GE`, `YIELD`, `COUNTY`,
`source_desc=SURVEY` — never the 5-yearly census); units kept as `BU / ACRE` (t/ha
conversion is a processing step); disclosure-suppressed `"(D)"` values become **NaN,
not dropped rows** (LINV-004); `"1,234"` thousands separators must be stripped.

**POWER:** AG-community units (T2M °C, RH2M %, ALLSKY_SFC_SW_DWN MJ m⁻² day⁻¹, WS2M
m s⁻¹); the daily record starts **1981**; missing days arrive as the fill value
**-999**, which must become NaN before any statistic touches it (a single leaked -999
drags a monthly mean below freezing); one pull = one ~50 km grid cell, so a "county"
series is a centroid approximation; **no rainfall here** — precipitation joins later
from CHIRPS, exactly as in production.

## 3. The shipped implementation

Displayed live from `src/` — never a copy.

In [ ]:
import inspect

from IPython.display import Code

from src.data.acquisition import (
    _build_power_url,
    _build_query_url,
    _power_to_frame,
    _records_to_frame,
    download_nass_yields,
    download_weather_data,
)

Code(
    inspect.getsource(_build_query_url)
    + inspect.getsource(_records_to_frame)
    + inspect.getsource(download_nass_yields),
    language="python",
)

In [ ]:
Code(
    inspect.getsource(_build_power_url)
    + inspect.getsource(_power_to_frame)
    + inspect.getsource(download_weather_data),
    language="python",
)

Walkthrough of the load-bearing lines:

- `_build_query_url` embeds the NASS API key, so nothing ever logs the URL; the key
  lives only in the `NASS_API_KEY` environment variable. POWER needs no key.
- Both `_records_to_frame` and `_power_to_frame` raise `ConnectionError` on error
  payloads — the APIs signal failure in-band (`data` missing / `properties.parameter`
  missing), not with HTTP codes alone.
- `download_weather_data` validates coordinates and the year window **before any
  network use** — bad inputs fail fast and cheap (and the curriculum's contract tests
  rely on exactly that).
- `replace(POWER_FILL_VALUE, nan)` implements the -999 rule from Section 2.

## 4. Worked example (offline, on the bundled samples)

In [ ]:
import json

nass_payload = json.loads((ROOT / "data/samples/nass_quickstats_sample.json").read_text(encoding="utf-8"))
yields = _records_to_frame(nass_payload)
yields

In [ ]:
# NASS provenance invariants: every row tagged; suppression became NaN, not a gap.
assert (yields["data_source"] == "api_nass").all()
assert yields["yield_value"].isna().sum() == 1

power_payload = json.loads((ROOT / "data/samples/nasa_power_sample.json").read_text(encoding="utf-8"))
weather = _power_to_frame(power_payload, latitude=42.03, longitude=-93.62)
weather

In [ ]:
# POWER provenance invariants: the -999 humidity became NaN (never a number), and the
# tag distinguishes this stream from NASS forever after.
assert weather["RH2M"].isna().sum() == 1
assert (weather["data_source"] == "api_nasa_power").all()
weather[["date", "T2M", "RH2M"]]

## 5. Pitfalls and connections

- **The 50,000-record NASS limit**: pull per crop; split by year range or state if a
  query grows past it.
- **Survey ≠ census** (NASS) and **reanalysis ≠ station** (POWER): mixing sources
  silently changes the error structure the Phase-2 models assume.
- **-999 is the classic silent poison**: one leaked fill value shifts means, inflates
  variances, and fabricates cold snaps. The replace-to-NaN happens at acquisition so
  no downstream week can forget it.
- **Connection to Phase 2**: NASS gives the `county × year` outcome table; POWER gives
  the daily series that becomes growing-season features (GDD, heat days, radiation
  sums) in the preprocessing weeks — then both meet in the hierarchical model (M2a).
- **Production parallel**: same tag-at-ingest discipline, same POWER API, same
  fallback thinking (POWER for meteorology, CHIRPS for rainfall) as
  keragita-farm-intelligence's weather pipeline.

---
**Provenance:** module `src/data/acquisition.py` @ 0.1.0 · deterministic (no RNG) ·
written 2026-07-07, extended 2026-07-08 · author Desmond Momanyi Mariita